# 03 — Fit and tune the models at three dates

**Stage:** modelling.

**Study design.** A bank does not train a model once and keep it for two years: it updates it as new calls come in. This notebook imitates that. The calls are cut into four time blocks, and models are trained at three successive dates, each time on **every call known at that date** (an "expanding window"):

| Window | Trained on | Settings chosen on | Tested on (notebook 04) |
|---|---|---|---|
| A — end of 2008 | May–Dec 2008 | Oct–Dec 2008 | 2009a, 2009b, 2010 |
| B — end of May 2009 | May 2008 – May 2009 | Mar–May 2009 | 2009b, 2010 |
| C — end of 2009 | May 2008 – Dec 2009 | Jun–Dec 2009 | 2010 |

For each window, two models are fitted (a logistic regression, the baseline, and a random forest, the flexible model) on two feature sets (with and without the economic context). That is 3 × 2 × 2 = 12 fitted models. Their predicted probabilities on the later blocks are saved for notebook 04. Nothing in this notebook looks at a block after the end of the window.

> **Origin.** This notebook was generated by Claude (Anthropic), a generative AI assistant, at the student's request; the study design and the modelling choices were discussed with the student. See "Declaration of tools" in `README.md`.

In [1]:
import sys
from pathlib import Path

# Make the shared modules in src/ importable from this notebook
sys.path.append(str(Path.cwd().parent))

In [2]:
import json

import pandas as pd
from sklearn.metrics import roc_auc_score

from src.data_utils import load_processed, PROCESSED_FILE, RESULTS_DIR
from src.modelling import (BLOCKS, WINDOWS, FEATURE_SETS, MODEL_NAMES, GRIDS, EXCLUDED,
                           add_block, calls_up_to, calls_between, make_model,
                           feature_columns, target_vector)

df = add_block(load_processed())
RESULTS_DIR.mkdir(exist_ok=True)
df.shape

(41188, 25)

## 1. The four time blocks

In [3]:
blocks = (df.groupby("block")
            .agg(first_month=("year_month", "min"), last_month=("year_month", "max"),
                 calls=("y", "size"), subscription_rate=("y", lambda s: (s == "yes").mean()))
            .loc[list(BLOCKS)])
blocks.round(3)

,first_month,last_month,calls,subscription_rate
block,,,,
2008,2008-05,2008-12,27690,0.048
2009a,2009-03,2009-05,8534,0.128
2009b,2009-06,2009-12,2906,0.391
2010,2010-03,2010-11,2058,0.521


The blocks are unequal: 2008 holds two thirds of the calls, and the bank called far fewer clients after the crisis. The subscription rate rises from 6% to about 50%.

## 2. Which columns the models use

The lists are defined once, in `src/modelling.py`.

| Columns | Decision | Reason |
|---|---|---|
| client and call characteristics (`age`, `job`, `contact`, …) | kept | Known before the call. |
| `previous`, `poutcome` (earlier campaigns) | kept | Empty in mid-2008 (nobody had been contacted before), but they fill up from late 2008, so windows B and C can learn from them. |
| `pdays` | left out | Coded 999 for "never contacted", which would distort a numeric scale; its information is already in `previous` and `poutcome`. |
| `month` | left out | Each window only contains some months, and later blocks contain months the model has never seen, so the month mostly tells *which period* a call is in. |
| `duration` | left out | Known only after the call (removed in notebook 01). |
| the five economic columns (`euribor3m`, …) | **kept** in the main models, removed in a second version | They carry the change of context, which is the heart of the question. The version without them is a robustness check. |

In [4]:
print("Left out of every model:", EXCLUDED)
for name in FEATURE_SETS:
    print(f"{name} ({len(feature_columns(name))} columns):", feature_columns(name))

Left out of every model: ['duration', 'month', 'pdays']
with_macro (17 columns): ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'day_of_week', 'poutcome', 'age', 'campaign', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']
without_macro (12 columns): ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'day_of_week', 'poutcome', 'age', 'campaign', 'previous']


How much contact history each window has to learn from:

In [5]:
pd.DataFrame({
    window: {"calls": len(calls_up_to(df, w["fit_end"])),
             "share_previously_contacted": (calls_up_to(df, w["fit_end"])["previous"] > 0).mean()}
    for window, w in WINDOWS.items()}).T.round(3)

,calls,share_previously_contacted
A_end2008,27690.0,0.027
B_may2009,36224.0,0.091
C_end2009,39130.0,0.113


## 3. Choosing the settings of each model

Each model has settings that are not learned from the data:
- **logistic regression:** `C`, the strength of the penalty on large coefficients (small `C` = strong penalty, simpler model);
- **random forest:** `max_depth`, how deep each tree may grow, and `min_samples_leaf`, the smallest group of calls a tree may end in. Both limit how closely the forest follows the training data.

For each window, every candidate is fitted on the earlier part of the window and scored on its most recent months (see the table at the top). The candidate with the highest **ROC-AUC** is kept. ROC-AUC measures how well the model *ranks* clients from most to least likely to subscribe; unlike accuracy, it does not depend on the share of clients who subscribe, which changes a lot over time.

The candidate lists are short on purpose: the project must run in the course workspace, and a few well-spaced values are enough to find the right order of magnitude.

In [6]:
rows = []
for window, w in WINDOWS.items():
    tune_fit = calls_up_to(df, w["tune_fit_end"])
    tune_valid = calls_between(df, *w["tune_valid"])
    for feature_set in FEATURE_SETS:
        cols = feature_columns(feature_set)
        for model_name in MODEL_NAMES:
            for params in GRIDS[model_name]:
                model = make_model(model_name, feature_set, **params)
                model.fit(tune_fit[cols], target_vector(tune_fit))
                proba = model.predict_proba(tune_valid[cols])[:, 1]
                rows.append({"window": window, "feature_set": feature_set, "model": model_name,
                             "params": json.dumps(params),
                             "tuning_roc_auc": roc_auc_score(target_vector(tune_valid), proba)})

tuning = pd.DataFrame(rows)
print(len(tuning), "candidates scored")

54 candidates scored


## 4. Table 3: the chosen settings

For each window, model and feature set: the candidate with the best ROC-AUC on the tuning months.

In [7]:
best = (tuning.sort_values("tuning_roc_auc", ascending=False)
              .groupby(["window", "feature_set", "model"], sort=False).head(1)
              .sort_values(["window", "feature_set", "model"])
              .reset_index(drop=True))
best.to_csv(RESULTS_DIR / "table3_chosen_settings.csv", index=False)
best.round(3)

,window,feature_set,model,params,tuning_roc_auc
0,A_end2008,with_macro,logistic_regression,"{""C"": 0.01}",0.486
1,A_end2008,with_macro,random_forest,"{""max_depth"": 12, ""min_samples_leaf"": 10}",0.425
2,A_end2008,without_macro,logistic_regression,"{""C"": 0.01}",0.442
3,A_end2008,without_macro,random_forest,"{""max_depth"": null, ""min_samples_leaf"": 10}",0.413
4,B_may2009,with_macro,logistic_regression,"{""C"": 0.1}",0.632
5,B_may2009,with_macro,random_forest,"{""max_depth"": 12, ""min_samples_leaf"": 10}",0.608
6,B_may2009,without_macro,logistic_regression,"{""C"": 0.1}",0.583
7,B_may2009,without_macro,random_forest,"{""max_depth"": 6, ""min_samples_leaf"": 10}",0.611
8,C_end2009,with_macro,logistic_regression,"{""C"": 1}",0.606
9,C_end2009,with_macro,random_forest,"{""max_depth"": null, ""min_samples_leaf"": 10}",0.598


**How to read the tuning scores.** For window A, the tuning months (Oct–Dec 2008) come right after the training months, and the scores are low: the 2008 data carry little information about who subscribes once `duration` is removed, and some relationships already change at the end of 2008 (for example, landline calls do better than mobile calls in late 2008, the reverse of mid-2008). For windows B and C, the scores are higher. Notebook 04 looks at this properly, on the test blocks.

*Write your own reading of Table 3 here.*

## 5. Fit the chosen models and save their predictions

Each chosen model is fitted on its whole window, then predicts a probability of subscribing for every call in the blocks after the window. The predictions are saved to `data/processed/predictions.csv`, one row per call, window, model and feature set.

In [8]:
parts = []
for _, row in best.iterrows():
    w = WINDOWS[row["window"]]
    cols = feature_columns(row["feature_set"])
    fit = calls_up_to(df, w["fit_end"])
    model = make_model(row["model"], row["feature_set"], **json.loads(row["params"]))
    model.fit(fit[cols], target_vector(fit))

    test = df[df["block"].isin(w["test_blocks"])]
    parts.append(pd.DataFrame({
        "call": test.index, "block": test["block"].values, "y": target_vector(test).values,
        "window": row["window"], "model": row["model"], "feature_set": row["feature_set"],
        "proba": model.predict_proba(test[cols])[:, 1].round(5)}))

predictions = pd.concat(parts, ignore_index=True)
predictions.to_csv(PROCESSED_FILE.parent / "predictions.csv", index=False)
print("Saved", len(predictions), "predictions")
# Number of test calls per window and block (each call is predicted by 4 models)
predictions.groupby(["window", "block"])["call"].nunique().unstack()

Saved 82080 predictions


block,2009a,2009b,2010
window,,,
A_end2008,8534.0,2906.0,2058.0
B_may2009,NaN,2906.0,2058.0
C_end2009,NaN,NaN,2058.0
